# Whale Tail Segment Anything Create Mask

https://github.com/facebookresearch/segment-anything<br/>
https://arxiv.org/abs/2304.02643<br/>
The Segment Anything (SA) project introduces a new task, model, and dataset for image segmentation. The Segment Anything Model (SAM) and corresponding dataset (SA-1B) are being released to foster research into foundation models for computer vision.

In [ ]:
!pip install git+https://github.com/facebookresearch/segment-anything.git

In [ ]:
import os
import cv2
import sys
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
from segment_anything import sam_model_registry 
from segment_anything import SamAutomaticMaskGenerator
from segment_anything import SamPredictor

In [ ]:
def show_anns(anns, axes=None):
    if len(anns) == 0:
        return
    if axes:
        ax = axes
    else:
        ax = plt.gca()
        ax.set_autoscale_on(False)
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    polygons = []
    color = []
    for ann in sorted_anns:
        m = ann['segmentation']
        img = np.ones((m.shape[0], m.shape[1], 3))
        color_mask = np.random.random((1, 3)).tolist()[0]
        for i in range(3):
            img[:,:,i] = color_mask[i]
        ax.imshow(np.dstack((img, m*0.5)))

def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 1.0]) #no transparency
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

    
def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)   

    
def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0,0,0,0), lw=2))    

# Fill the background with mask other than the tail

In [ ]:
path0='/kaggle/input/humpback-whale-identification/test/00744bd58.jpg'
image = cv2.imread(path0)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
image = cv2.resize(image,dsize=(200,200))
plt.figure(figsize=(4,4))
plt.imshow(image)
plt.axis('off')
plt.show()

# SAM model

In [ ]:
sam_checkpoint = "/kaggle/input/segment-anything-models/sam_vit_h_4b8939.pth"
model_type = "vit_h"#
device = "cpu"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)
predictor = SamPredictor(sam)
predictor.set_image(image)

# When set label 0 (fish) and label 1 (background)
### This method is available when the tail position is known

In [ ]:
print(image.shape)

input_point = np.array([[5,5],[195,195],[5,195],[195,5],[100,120],[110,110]])
input_label = np.array([1,1,1,1,0,0]) # label 1 (green) segmented, label 0 (red) excluded

plt.figure(figsize=(4,4))
plt.imshow(image)
show_points(input_point, input_label, plt.gca())
plt.axis('on')
plt.title('Original', fontsize=12)
plt.show()  

masks, scores, logits = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True,
)

for i, (mask, score) in enumerate(zip(masks, scores)):
    plt.figure(figsize=(4,4))
    plt.imshow(image)
    show_mask(mask, plt.gca())
    #show_points(input_point, input_label, plt.gca())
    plt.title(f"Mask {i+1}, Score: {score:.3f}", fontsize=12)
    plt.show()  
  

# When set only label 0 (background)
### This method is availbale even when the tail position is unknwon 

In [ ]:
input_point2 = np.array([[5,5],[195,195],[5,195],[195,5]])
input_label2 = np.array([1,1,1,1]) # label 1 (green) segmented, label 0 (red) excluded

plt.figure(figsize=(4,4))
plt.imshow(image)
show_points(input_point2, input_label2, plt.gca())
plt.axis('on')
plt.title('Original', fontsize=12)
plt.show()  

masks2, scores2, logits2 = predictor.predict(
    point_coords=input_point2,
    point_labels=input_label2,
    multimask_output=True,
)

    
for i, (mask, score) in enumerate(zip(masks2, scores2)):
    plt.figure(figsize=(4,4))
    plt.imshow(image)
    show_mask(mask, plt.gca())
    #show_points(input_point, input_label, plt.gca())
    plt.title(f"Mask {i+1}, Score: {score:.3f}", fontsize=12)
    plt.savefig(f"mask{i+1}.png")
    plt.show()    